In [ ]:
# ═══ A.1 — GPU ═══
import torch
print(f"CUDA : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    cc = torch.cuda.get_device_capability(0)
    print(f"GPU  : {torch.cuda.get_device_name(0)}")
    print(f"CC   : {cc[0]}.{cc[1]}")
    assert cc[0] >= 7, "❌ GPU trop ancien"
print("✅ A.1 PASSÉ")

In [ ]:
# ═══ A.2 — Installation Universelle (Auto-détection GPU & RAM) ═══
import os, sys, glob, shutil, site, torch, multiprocessing

assert torch.cuda.is_available(), "❌ Aucun GPU détecté."

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
target_arch = f"{major}.{minor}"

print(f"✅ GPU détecté : {gpu_name}")
print(f"🎯 Architecture ciblée : sm_{major}{minor} ({target_arch})")

!apt-get install -y -q ninja-build

import psutil
total_ram_gb = psutil.virtual_memory().total / (1024 ** 3)
max_jobs = 1 if total_ram_gb < 16.0 else min(2, multiprocessing.cpu_count())

os.environ['CUDA_HOME'] = '/usr/local/cuda'
os.environ['FORCE_CUDA'] = '1'
os.environ['TORCH_CUDA_ARCH_LIST'] = target_arch
os.environ['MAX_JOBS'] = str(max_jobs)

print(f"⚙️  Compilation configurée : MAX_JOBS={max_jobs} (RAM: {total_ram_gb:.1f} Go)")

# Dépendances Python — versions figées pour reproductibilité
!pip install -q timm==1.0.29 scipy opencv-python pycocotools tensorboard tensorboardX fairscale einops
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

if not os.path.exists('/content/detrex'):
    !git clone https://github.com/IDEA-Research/detrex.git /content/detrex
!rm -rf /content/detrex/detectron2

%cd /content/detrex
!rm -rf build/ detrex/_C*.so detrex.egg-info
!MAX_JOBS={max_jobs} CUDA_HOME=/usr/local/cuda FORCE_CUDA=1 TORCH_CUDA_ARCH_LIST="{target_arch}" python setup.py build_ext --inplace
%cd /content

so = glob.glob('/content/detrex/detrex/_C*.so')
assert so, f"❌ Échec : le fichier .so pour {target_arch} n'a pas été produit."
print(f"✅ Extension CUDA compilée : {os.path.basename(so[0])}")

sp = site.getsitepackages()[0]
for pat in ('detrex', 'detrex.egg-link', 'detrex.egg-info'):
    for p in glob.glob(os.path.join(sp, pat)):
        shutil.rmtree(p, ignore_errors=True) if os.path.isdir(p) else os.remove(p)

with open(os.path.join(sp, 'zz_detrex_local.pth'), 'w') as f:
    f.write('/content/detrex\n')

if '/content/detrex' not in sys.path:
    sys.path.insert(0, '/content/detrex')

print("\n🚀 Installation terminée.")
print("👉 Redémarrez la session (Exécution > Redémarrer la session).")

In [ ]:
# ═══ A.3-bis — Constantes idempotentes (safe après restart) ═══
import os, json

_PARAMS = '/content/train_params.json'

if not os.path.exists(_PARAMS):
    # A.4 doit être exécutée en premier
    print("⚠️  train_params.json absent — exécutez A.4 d'abord.")
else:
    with open(_PARAMS) as f:
        _p = json.load(f)
    # Injecte/rafraîchit les constantes critiques dans l'espace de noms
    globals().update({
        'DINOV2_CHECKPOINT': _p['DINOV2_CHECKPOINT'],
        'TRAIN_JSON_DRIVE':  _p['TRAIN_JSON_DRIVE'],
        'TEST_JSON_DRIVE':   _p['TEST_JSON_DRIVE'],
        'IMAGES_DIR_LOCAL':  _p['LOCAL_IMAGES'],
        'OUTPUT_DIR':        _p['OUTPUT_DIR'],
        'TRAIN_PARAMS_FILE': _PARAMS,
    })
    print(f"✅ Constantes restaurées : {DINOV2_CHECKPOINT}")
print("✅ A.3-bis PASSÉ")

In [ ]:
# ═══ A.4 — Drive + params (patché v2) ═══
import os, json
import numpy as np                                   # ← FIX : np manquant
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT        = '/content/drive/MyDrive/EMBED_Dataset'
DINOV2_CHECKPOINT = os.path.join(DRIVE_ROOT, 'checkpoints', 'dinov2_latest_checkpoint.pth')
LOCAL_DATA_ROOT   = '/content/mammo_data'
IMAGES_DIR_LOCAL  = os.path.join(LOCAL_DATA_ROOT, 'images')
ZIP_PATH_DRIVE    = os.path.join(DRIVE_ROOT, 'curated', 'full_dataset', 'images.zip')
ZIP_PATH_LOCAL    = '/content/images.zip'
TRAIN_JSON_DRIVE  = os.path.join(DRIVE_ROOT, 'curated', 'full_dataset', 'coco', 'mass_train.json')
TEST_JSON_DRIVE   = os.path.join(DRIVE_ROOT, 'curated', 'full_dataset', 'coco', 'mass_val.json')
OUTPUT_DIR        = os.path.join(DRIVE_ROOT, 'curated', 'full_dataset', 'RF_DETR')
TRAIN_PARAMS_FILE = '/content/train_params.json'

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOCAL_DATA_ROOT, exist_ok=True)
os.makedirs('/content/mammo/configs', exist_ok=True)

# --- Hyperparamètres ---
BATCH_SIZE        = 2
GRAD_ACCUM_STEPS  = 8
NUM_WORKERS       = 1                                # ← FIX : Colab + cv2 → 1 (au lieu de 2)
NUM_EPOCHS        = 12
LOG_PERIOD        = 20
LEARNING_RATE     = 1e-4
BACKBONE_LR       = 3e-5
LAYER_DECAY       = 0.90
WEIGHT_DECAY      = 1e-4
LR_DROP_MILESTONES_PCT = [0.70, 0.90]
GAMMA             = 0.1

DINOV2_MODEL_NAME = "vit_small_patch14_dinov2.lvd142m"
FREEZE_BLOCKS     = 8
PATCH_SIZE        = 14

NUM_QUERIES       = 50
NUM_DN_QUERIES    = 6

RF_SCALE_FACTORS  = (2.0, 1.0, 0.5, 0.25)
RF_NUM_BLOCKS     = 3
RF_SURVIVAL_PROB  = 1.0
RF_FORCE_DROP_N   = 0

IMAGE_SHORT_EDGES = (644, 672, 700, 728, 756, 784, 812)
IMAGE_MAX_SIZE    = 1624
IMAGE_TEST_SIZE   = 812

AMP_ENABLED       = False
# ← FIX : fallback automatique si bf16 non supporté nativement (V100, T4)
if not torch.cuda.is_bf16_supported():
    AMP_DTYPE = 'float16'
    print("⚠️  bf16 non supporté nativement → fallback AMP_DTYPE='float16'")
else:
    AMP_DTYPE = 'bfloat16'
print(f"⚡ AMP_DTYPE = {AMP_DTYPE}")

GRAD_CLIP_NORM    = 0.1
SCORE_THRESH      = 0.20
TB_IMAGE_PERIOD   = 500

# --- Calcul iter_per_epoch SANS fallback ---
with open(TRAIN_JSON_DRIVE) as f:
    num_train_images = len(json.load(f)['images'])
assert num_train_images > 0, f"❌ Aucune image dans {TRAIN_JSON_DRIVE}"

iters_per_epoch = max(1, int(np.ceil(num_train_images / BATCH_SIZE)))
MAX_ITER = int(NUM_EPOCHS * iters_per_epoch)
WARMUP_ITERS = int(0.02 * MAX_ITER)
CHECKPOINT_PERIOD = max(1, iters_per_epoch)

MILESTONES = sorted({int(MAX_ITER * p) for p in LR_DROP_MILESTONES_PCT
                     if 0.0 < p < 1.0})
MILESTONES = [m for m in MILESTONES if 0 < m < MAX_ITER]
assert len(MILESTONES) == len(LR_DROP_MILESTONES_PCT), (
    f"❌ Milestones tronqués : {MILESTONES} vs {LR_DROP_MILESTONES_PCT} "
    f"(MAX_ITER={MAX_ITER})"
)

train_params = {
    'BATCH_SIZE': BATCH_SIZE, 'NUM_WORKERS': NUM_WORKERS, 'NUM_EPOCHS': NUM_EPOCHS,
    'GRAD_ACCUM_STEPS': GRAD_ACCUM_STEPS,
    'MAX_ITER': MAX_ITER, 'ITERS_PER_EPOCH': iters_per_epoch,
    'CHECKPOINT_PERIOD': CHECKPOINT_PERIOD, 'MAX_CHECKPOINTS_TO_KEEP': 6,
    'LOG_PERIOD': LOG_PERIOD,
    'LEARNING_RATE': LEARNING_RATE, 'BACKBONE_LR': BACKBONE_LR,
    'LAYER_DECAY': LAYER_DECAY, 'WEIGHT_DECAY': WEIGHT_DECAY,
    'WARMUP_ITERS': WARMUP_ITERS,
    'MILESTONES': MILESTONES, 'GAMMA': GAMMA, 'GRAD_CLIP_NORM': GRAD_CLIP_NORM,
    'RF_SCALE_FACTORS': list(RF_SCALE_FACTORS),
    'RF_NUM_BLOCKS': RF_NUM_BLOCKS,
    'RF_SURVIVAL_PROB': RF_SURVIVAL_PROB,
    'RF_FORCE_DROP_N': RF_FORCE_DROP_N,
    'SCORE_THRESH': SCORE_THRESH, 'TB_IMAGE_PERIOD': TB_IMAGE_PERIOD,
    'IMAGE_MAX_SIZE': IMAGE_MAX_SIZE, 'IMAGE_SHORT_EDGES': list(IMAGE_SHORT_EDGES),
    'IMAGE_TEST_SIZE': IMAGE_TEST_SIZE, 'AMP_ENABLED': AMP_ENABLED,
    'AMP_DTYPE': AMP_DTYPE,
    'DINOV2_MODEL_NAME': DINOV2_MODEL_NAME, 'FREEZE_BLOCKS': FREEZE_BLOCKS,
    'DINOV2_CHECKPOINT': DINOV2_CHECKPOINT, 'PATCH_SIZE': PATCH_SIZE,
    'NUM_QUERIES': NUM_QUERIES, 'NUM_DN_QUERIES': NUM_DN_QUERIES,
    'OUTPUT_DIR': OUTPUT_DIR,
    'TRAIN_JSON_DRIVE': TRAIN_JSON_DRIVE, 'TEST_JSON_DRIVE': TEST_JSON_DRIVE,
    'LOCAL_IMAGES': IMAGES_DIR_LOCAL,
}
with open(TRAIN_PARAMS_FILE, 'w') as f:
    json.dump(train_params, f, indent=2)

print(f"✅ {num_train_images:,} images → {iters_per_epoch:,} it/ép × {NUM_EPOCHS} = {MAX_ITER:,}")
print(f"   MILESTONES = {MILESTONES}")
print(f"   BATCH_SIZE = {BATCH_SIZE} | GRAD_ACCUM = {GRAD_ACCUM_STEPS} "
      f"→ batch effectif = {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"   AMP = {AMP_ENABLED} ({AMP_DTYPE})")
assert os.path.exists(DINOV2_CHECKPOINT), f"❌ {DINOV2_CHECKPOINT}"
assert os.path.exists(TRAIN_JSON_DRIVE), f"❌ {TRAIN_JSON_DRIVE}"
print("✅ A.4 PASSÉ")

In [ ]:
# ═══ A.5 — Dataset ═══
import zipfile, shutil, glob, time

def _flatten_nested(nested, target):
    if not os.path.isdir(nested): return
    for f in os.listdir(nested):
        src, dst = os.path.join(nested, f), os.path.join(target, f)
        if not os.path.exists(dst): shutil.move(src, dst)
        else:
            try: os.remove(src)
            except: pass
    try: os.rmdir(nested)
    except: pass

_flatten_nested(os.path.join(IMAGES_DIR_LOCAL, 'images'), IMAGES_DIR_LOCAL)
existing = glob.glob(os.path.join(IMAGES_DIR_LOCAL, '*.png'))

if len(existing) < 4000:
    assert os.path.exists(ZIP_PATH_DRIVE), f"❌ ZIP : {ZIP_PATH_DRIVE}"
    if os.path.exists(ZIP_PATH_LOCAL) and not zipfile.is_zipfile(ZIP_PATH_LOCAL):
        os.remove(ZIP_PATH_LOCAL)
    if not os.path.exists(ZIP_PATH_LOCAL):
        print(f"📦 Copie ZIP..."); t0 = time.time()
        shutil.copyfile(ZIP_PATH_DRIVE, ZIP_PATH_LOCAL)
        print(f"   ✅ {time.time()-t0:.1f}s")
    print(f"📂 Décompression..."); t0 = time.time()
    with zipfile.ZipFile(ZIP_PATH_LOCAL) as z:
        names = z.namelist()
        tgt = LOCAL_DATA_ROOT if any(n.startswith('images/') for n in names[:50]) else IMAGES_DIR_LOCAL
        z.extractall(tgt)
    _flatten_nested(os.path.join(IMAGES_DIR_LOCAL, 'images'), IMAGES_DIR_LOCAL)

existing = glob.glob(os.path.join(IMAGES_DIR_LOCAL, '*.png'))
assert len(existing) >= 4000, f"❌ Seulement {len(existing)} PNG"
print(f"✅ {len(existing):,} PNG dans {IMAGES_DIR_LOCAL}")
print("✅ A.5 PASSÉ")

In [ ]:
# ═══ A.6 — Package structure ═══
import os
os.makedirs('/content/mammo/configs', exist_ok=True)
for p in ['/content/mammo/__init__.py', '/content/mammo/configs/__init__.py']:
    open(p, 'w').close()

import sys, importlib
sys.path.insert(0, '/content')
import mammo
print(f"✅ mammo.__file__ = {mammo.__file__}")
print("✅ A.6 PASSÉ")

In [ ]:
# ═══ B.1 — backbone.py (patché) ═══
%%writefile /content/mammo/backbone.py
"""DINOv2 ViT-S/14 — extraction multi-blocs {3, 6, 9, 12}."""
import os
import timm
import torch
from detectron2.layers import ShapeSpec
from detectron2.modeling.backbone import Backbone


class DINOv2MultiScaleBackbone(Backbone):
    BLOCK_INDICES = (2, 5, 8, 11)

    def __init__(self, checkpoint_path=None,
                 model_name="vit_small_patch14_dinov2.lvd142m",
                 freeze_blocks=2,
                 out_features=("block3", "block6", "block9", "block12")):
        super().__init__()
        self.vit = timm.create_model(
            model_name, pretrained=False,
            dynamic_img_size=True, dynamic_img_pad=True, num_classes=0,
        )
        self._p = int(self.vit.patch_embed.patch_size[0])
        self._dim = int(self.vit.embed_dim)
        self.n_blocks = len(self.vit.blocks)
        self.out_features = list(out_features)
        self._out_channels = {n: self._dim for n in self.out_features}
        self._out_strides = {n: self._p for n in self.out_features}

        if checkpoint_path:
            self._load_teacher(checkpoint_path)
        self.freeze_blocks = int(freeze_blocks)
        self._apply_partial_freeze()

    def _load_teacher(self, path):
        if not os.path.isfile(path):
            raise FileNotFoundError(f"Checkpoint : {path}")
        try:
            ckpt = torch.load(path, map_location="cpu", weights_only=False)
        except TypeError:
            ckpt = torch.load(path, map_location="cpu")

        if "teacher_backbone" in ckpt:
            sd = ckpt["teacher_backbone"]
        elif "model_state_dict" in ckpt:
            sd = {k: v for k, v in ckpt["model_state_dict"].items()
                  if k.startswith("teacher_backbone.")}
        else:
            sd = {k: v for k, v in ckpt.items() if "student" not in k.lower()}

        prefixes = ("teacher_backbone.vit.", "teacher_backbone.", "vit.", "backbone.")
        clean = {}
        for k, v in sd.items():
            for pre in prefixes:
                if k.startswith(pre):
                    k = k[len(pre):]; break
            if not k.startswith(("head", "dino_head", "ibot_head")):
                clean[k] = v
        clean = {k: v for k, v in clean.items() if k != "mask_token"}

        missing, unexpected = self.vit.load_state_dict(clean, strict=False)
        print(f"📦 [DINOv2] {len(clean)} tenseurs chargés "
              f"({len(missing)} manquants, {len(unexpected)} inattendus)")
        if len(clean) < 100:
            raise RuntimeError(f"❌ Trop peu de tenseurs ({len(clean)})")

    def _apply_partial_freeze(self):
        for p in self.vit.parameters():
            p.requires_grad = False
        for i, blk in enumerate(self.vit.blocks):
            if i >= self.freeze_blocks:
                for p in blk.parameters():
                    p.requires_grad = True
        for p in self.vit.norm.parameters():
            p.requires_grad = True
        n_t = sum(p.numel() for p in self.vit.parameters() if p.requires_grad)
        n_a = sum(p.numel() for p in self.vit.parameters())
        print(f"🔓 [DINOv2] Freeze {self.freeze_blocks}/{self.n_blocks} | "
              f"{n_t/1e6:.2f}M/{n_a/1e6:.2f}M")

    def forward(self, x):
        intermediates = self.vit.forward_intermediates(
            x, indices=self.BLOCK_INDICES,
            output_fmt='NCHW', intermediates_only=True,
        )
        return {name: feat.contiguous()
                for name, feat in zip(self.out_features, intermediates)}

    def output_shape(self):
        return {n: ShapeSpec(channels=self._out_channels[n],
                             stride=self._out_strides[n])
                for n in self.out_features}

    @property
    def size_divisibility(self):
        return self._p * 4

In [ ]:
# ═══ B.2 — Test backbone ═══
import sys, importlib, torch
sys.path.insert(0, '/content')
for m in list(sys.modules.keys()):
    if m.startswith('mammo'): del sys.modules[m]
import mammo.backbone; importlib.reload(mammo.backbone)
from mammo.backbone import DINOv2MultiScaleBackbone

print("=" * 60); print("B.2 — Test DINOv2MultiScaleBackbone"); print("=" * 60)
bb = DINOv2MultiScaleBackbone(checkpoint_path=DINOV2_CHECKPOINT,
                              freeze_blocks=2).cuda()

frozen = [n for n, p in bb.named_parameters() if not p.requires_grad]
for n in frozen:
    assert not n.startswith(tuple(f'vit.blocks.{i}.' for i in range(2, 12))), \
        f"❌ {n} ne devrait pas être gelé"
print("  ✅ Blocs 0-1 gelés, 2-11 dégelés")

for H, W in [(812, 812), (700, 1000)]:
    with torch.no_grad():
        out = bb(torch.randn(1, 3, H, W, device='cuda'))
    assert set(out.keys()) == {"block3", "block6", "block9", "block12"}
    Hp = (H + 13) // 14
    Wp = (W + 13) // 14
    for name, feat in out.items():
        assert feat.shape == (1, 384, Hp, Wp), \
            f"{name} : attendu (1, 384, {Hp}, {Wp}), obtenu {tuple(feat.shape)}"
    print(f"  ({H},{W}) → {Hp}×{Wp} ✅")

stacked = torch.stack([out[k] for k in sorted(out.keys())])
diff = (stacked[0] - stacked[-1]).abs().mean().item()
assert diff > 1e-3, f"❌ block3 et block12 quasi identiques (diff={diff})"
print(f"  ✅ block3 vs block12 : diff={diff:.4f}")

bb.train()
out = bb(torch.randn(1, 3, 812, 812, device='cuda'))
sum(f.sum() for f in out.values()).backward()
has_grad = any(p.grad is not None and p.grad.abs().sum() > 0
               for p in bb.vit.blocks[2].parameters())
assert has_grad, "❌ Pas de gradient au bloc 3"
print("  ✅ Bloc 3 reçoit un gradient")

frozen_grad = any(p.grad is not None and p.grad.abs().sum() > 0
                  for p in bb.vit.blocks[0].parameters())
assert not frozen_grad, "❌ Bloc 1 ne devrait pas avoir de gradient"
print("  ✅ Bloc 1 (gelé) : pas de gradient")
print("\n✅ B.2 PASSÉ")

In [ ]:
# ═══ C.1 — Écriture rfdetr_projector.py ═══
%%writefile /content/mammo/rfdetr_projector.py
"""
Projecteur multi-échelle RF-DETR officiel.
Source : https://github.com/roboflow/rf-detr/blob/853b5f3.../rfdetr/models/backbone/projector.py
Adapté pour DINOv2 ViT-S/14 (in_channels = liste de 4×384).
"""
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


class LayerNorm(nn.Module):
    """LayerNorm sur (B, C, H, W) — ConvNeXt style."""
    def __init__(self, normalized_shape, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(normalized_shape))
        self.bias = nn.Parameter(torch.zeros(normalized_shape))
        self.eps = eps
        self.normalized_shape = (normalized_shape,)

    def forward(self, x):
        u = x.mean(1, keepdim=True)
        s = (x - u).pow(2).mean(1, keepdim=True)
        x = (x - u) / torch.sqrt(s + self.eps)
        return self.weight[:, None, None] * x + self.bias[:, None, None]


def get_norm(norm, out_channels):
    if norm is None:
        return None
    if isinstance(norm, str):
        if len(norm) == 0:
            return None
        norm = {"LN": lambda ch: LayerNorm(ch)}[norm]
    return norm(out_channels)


def get_activation(name, inplace=False):
    if name == "silu":
        return nn.SiLU(inplace=inplace)
    if name == "relu":
        return nn.ReLU(inplace=inplace)
    if name in ["LeakyReLU", 'leakyrelu', 'lrelu']:
        return nn.LeakyReLU(0.1, inplace=inplace)
    if name is None:
        return nn.Identity()
    raise AttributeError(f"Unsupported act type: {name}")


class ConvX(nn.Module):
    """Conv + Norm + Activation (style YOLO/RF-DETR)."""
    def __init__(self, in_planes, out_planes, kernel=3, stride=1, groups=1,
                 dilation=1, act='relu', layer_norm=False, rms_norm=False):
        super().__init__()
        if not isinstance(kernel, tuple):
            kernel = (kernel, kernel)
        padding = (kernel[0] // 2, kernel[1] // 2)
        self.conv = nn.Conv2d(in_planes, out_planes, kernel_size=kernel,
                              stride=stride, padding=padding, groups=groups,
                              dilation=dilation, bias=False)
        if rms_norm:
            self.bn = nn.RMSNorm(out_planes)
        else:
            self.bn = get_norm('LN', out_planes) if layer_norm \
                      else nn.BatchNorm2d(out_planes)
        self.act = get_activation(act, inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))


class Bottleneck(nn.Module):
    def __init__(self, c1, c2, shortcut=True, g=1, k=(3, 3), e=0.5,
                 act='silu', layer_norm=False, rms_norm=False):
        super().__init__()
        c_ = int(c2 * e)
        self.cv1 = ConvX(c1, c_, k[0], 1, act=act,
                         layer_norm=layer_norm, rms_norm=rms_norm)
        self.cv2 = ConvX(c_, c2, k[1], 1, groups=g, act=act,
                         layer_norm=layer_norm, rms_norm=rms_norm)
        self.add = shortcut and c1 == c2

    def forward(self, x):
        return x + self.cv2(self.cv1(x)) if self.add else self.cv2(self.cv1(x))


class C2f(nn.Module):
    """CSP Bottleneck with 2 convolutions (RF-DETR)."""
    def __init__(self, c1, c2, n=1, shortcut=False, g=1, e=0.5,
                 act='silu', layer_norm=False, rms_norm=False):
        super().__init__()
        self.c = int(c2 * e)
        self.cv1 = ConvX(c1, 2 * self.c, 1, 1, act=act,
                         layer_norm=layer_norm, rms_norm=rms_norm)
        self.cv2 = ConvX((2 + n) * self.c, c2, 1, act=act,
                         layer_norm=layer_norm, rms_norm=rms_norm)
        self.m = nn.ModuleList(
            Bottleneck(self.c, self.c, shortcut, g, k=(3, 3), e=1.0,
                       act=act, layer_norm=layer_norm, rms_norm=rms_norm)
            for _ in range(n)
        )

    def forward(self, x):
        y = list(self.cv1(x).split((self.c, self.c), 1))
        y.extend(m(y[-1]) for m in self.m)
        return self.cv2(torch.cat(y, 1))


class MultiScaleProjector(nn.Module):
    """
    Projecteur multi-échelle RF-DETR (officiel).

    Args:
        in_channels: LISTE des canaux d'entrée (ex: [384, 384, 384, 384])
        out_channels: canaux de sortie (256)
        scale_factors: (2.0, 1.0, 0.5, 0.25) → p2, p3, p4, p5
        num_blocks: nombre de Bottleneck dans chaque C2f
        survival_prob: probabilité de survie des features (1.0 = pas de dropout)
        force_drop_last_n_features: force la mise à zéro des N dernières features
    """
    def __init__(self, in_channels, out_channels, scale_factors,
                 num_blocks=3, layer_norm=False, rms_norm=False,
                 survival_prob=1.0, force_drop_last_n_features=0):
        super().__init__()
        self.scale_factors = list(scale_factors)
        self.survival_prob = survival_prob
        self.force_drop_last_n_features = force_drop_last_n_features

        stages_sampling = []
        stages = []
        self.use_extra_pool = False

        for scale in self.scale_factors:
            if scale == 0.25:
                self.use_extra_pool = True
                continue

            stages_sampling.append([])
            for in_dim in in_channels:
                out_dim = in_dim
                layers = []
                if scale == 4.0:
                    layers.extend([
                        nn.ConvTranspose2d(in_dim, in_dim // 2, 2, stride=2),
                        get_norm('LN', in_dim // 2), nn.GELU(),
                        nn.ConvTranspose2d(in_dim // 2, in_dim // 4, 2, stride=2),
                    ])
                    out_dim = in_dim // 4
                elif scale == 2.0:
                    layers.extend([
                        nn.ConvTranspose2d(in_dim, in_dim // 2, 2, stride=2),
                    ])
                    out_dim = in_dim // 2
                elif scale == 1.0:
                    pass
                elif scale == 0.5:
                    layers.extend([
                        ConvX(in_dim, in_dim, 3, 2, layer_norm=layer_norm),
                    ])
                else:
                    raise NotImplementedError(f"Unsupported scale_factor: {scale}")

                stages_sampling[-1].append(nn.Sequential(*layers))

            stages_sampling[-1] = nn.ModuleList(stages_sampling[-1])

            in_dim = int(sum(ic // max(1, scale) for ic in in_channels))
            layers = [
                C2f(in_dim, out_channels, num_blocks,
                    layer_norm=layer_norm, rms_norm=rms_norm),
                get_norm('LN', out_channels),
            ]
            stages.append(nn.Sequential(*layers))

        self.stages_sampling = nn.ModuleList(stages_sampling)
        self.stages = nn.ModuleList(stages)

    def forward(self, x):
        num_features = len(x)

        if self.survival_prob < 1.0 and self.training:
            final_drop_prob = 1 - self.survival_prob
            drop_p = np.random.uniform()
            for i in range(1, num_features):
                critical_drop_prob = i * (final_drop_prob / (num_features - 1))
                if drop_p < critical_drop_prob:
                    x[i][:] = 0
        elif self.force_drop_last_n_features > 0:
            for i in range(self.force_drop_last_n_features):
                x[-(i + 1)] = torch.zeros_like(x[-(i + 1)])

        results = []
        for i, stage in enumerate(self.stages):
            feat_fuse = []
            for j, stage_sampling in enumerate(self.stages_sampling[i]):
                feat_fuse.append(stage_sampling(x[j]))
            feat_fuse = torch.cat(feat_fuse, dim=1) if len(feat_fuse) > 1 \
                        else feat_fuse[0]
            results.append(stage(feat_fuse))

        if self.use_extra_pool:
            results.append(F.max_pool2d(results[-1],
                                        kernel_size=1, stride=2, padding=0))
        return results


class BackboneProjectorWrapper(nn.Module):
    """Enveloppe {DINOv2MultiScaleBackbone + MultiScaleProjector}."""

    def __init__(self, backbone, projector):
        super().__init__()
        self.backbone = backbone
        self.projector = projector

        # ← FIX 7 : dérivé du backbone, plus hardcodé
        self._base_stride = backbone._p // 2  # patch_size / 2

        n_stages = len(projector.stages)
        self.out_names = [f"p{i + 2}" for i in range(n_stages)]
        self.out_strides = {
            name: self._base_stride * (2 ** i)
            for i, name in enumerate(self.out_names)
        }
        if projector.use_extra_pool:
            extra_name = f"p{n_stages + 2}"
            self.out_names.append(extra_name)
            self.out_strides[extra_name] = self.out_strides[self.out_names[-2]] * 2

    def forward(self, x):
        features = self.backbone(x)
        feat_list = [features[n] for n in self.backbone.out_features]
        pyramid = self.projector(feat_list)
        return {name: feat for name, feat in zip(self.out_names, pyramid)}

    def output_shape(self):
        from detectron2.layers import ShapeSpec
        return {name: ShapeSpec(channels=256, stride=stride)
                for name, stride in self.out_strides.items()}

    @property
    def size_divisibility(self):
        return self.backbone.size_divisibility

In [ ]:
# ═══ C.2 — Test LayerNorm + ConvX ═══
import torch, torch.nn as nn, sys, importlib
sys.path.insert(0, '/content')
for m in list(sys.modules.keys()):
    if m.startswith('mammo'): del sys.modules[m]
from mammo.rfdetr_projector import LayerNorm, ConvX

print("=" * 60); print("C.2 — Test LayerNorm + ConvX"); print("=" * 60)

x = torch.randn(2, 64, 16, 16)
ln = LayerNorm(64); y = ln(x)
assert y.shape == x.shape
ln_ref = nn.LayerNorm(64, eps=1e-6)
ln_ref.weight.data.copy_(ln.weight.data)
ln_ref.bias.data.copy_(ln.bias.data)
y_ref = ln_ref(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)
diff = (y - y_ref).abs().max().item()
assert diff < 1e-5, f"LayerNorm écart {diff}"
print(f"  LayerNorm écart vs nn.LayerNorm : {diff:.2e} ✅")

cv = ConvX(64, 128, 3, 1, act='silu')
y = cv(torch.randn(2, 64, 16, 16))
assert y.shape == (2, 128, 16, 16)
print(f"  ConvX 64→128 stride 1 : {tuple(y.shape)} ✅")

cv2 = ConvX(64, 64, 3, 2, act='silu')
y2 = cv2(torch.randn(2, 64, 16, 16))
assert y2.shape == (2, 64, 8, 8)
print(f"  ConvX 64→64 stride 2 : {tuple(y2.shape)} ✅")
print("\n✅ C.2 PASSÉ")

In [ ]:
# ═══ C.3 — Test Bottleneck ═══
import torch, sys, importlib
sys.path.insert(0, '/content')
for m in list(sys.modules.keys()):
    if m.startswith('mammo'): del sys.modules[m]
from mammo.rfdetr_projector import Bottleneck

print("=" * 60); print("C.3 — Test Bottleneck"); print("=" * 60)
for c1, c2, sc, e in [(64, 64, True, 1.0),
                      (32, 64, False, 1.0),
                      (64, 64, True, 0.5)]:
    blk = Bottleneck(c1, c2, shortcut=sc, e=e)
    x = torch.randn(2, c1, 16, 16, requires_grad=True)
    y = blk(x); y.sum().backward()
    assert y.shape == (2, c2, 16, 16)
    assert x.grad is not None
    print(f"  c1={c1} c2={c2} sc={sc} e={e} → {tuple(y.shape)} ✅")
print("\n✅ C.3 PASSÉ")

In [ ]:
# ═══ C.4 — Test C2f ═══
import torch, sys, importlib
sys.path.insert(0, '/content')
for m in list(sys.modules.keys()):
    if m.startswith('mammo'): del sys.modules[m]
from mammo.rfdetr_projector import C2f

print("=" * 60); print("C.4 — Test C2f"); print("=" * 60)
for c1, c2, n in [(1024, 256, 3), (768, 128, 2)]:
    blk = C2f(c1, c2, n=n, e=0.5)
    x = torch.randn(2, c1, 32, 32, requires_grad=True)
    y = blk(x); y.sum().backward()
    assert y.shape == (2, c2, 32, 32), f"attendu (2,{c2},32,32) obtenu {tuple(y.shape)}"
    assert x.grad is not None
    print(f"  {c1}→{c2} n={n} : {tuple(y.shape)} ✅")
print("\n✅ C.4 PASSÉ")

In [ ]:
# ═══ C.5 — Test MultiScaleProjector ═══
import torch, sys, importlib
sys.path.insert(0, '/content')
for m in list(sys.modules.keys()):
    if m.startswith('mammo'): del sys.modules[m]
from mammo.rfdetr_projector import MultiScaleProjector

print("=" * 60); print("C.5 — Test MultiScaleProjector"); print("=" * 60)

proj = MultiScaleProjector(
    in_channels=[384, 384, 384, 384],
    out_channels=256,
    scale_factors=(2.0, 1.0, 0.5, 0.25),
    num_blocks=3,
).cuda()

B, Hp, Wp = 1, 58, 58
x = [torch.randn(B, 384, Hp, Wp, device='cuda', requires_grad=True)
     for _ in range(4)]

out = proj(x)
print(f"  Nombre de niveaux : {len(out)}")
assert len(out) == 4, f"4 niveaux attendus, obtenu {len(out)}"

for i, feat in enumerate(out):
    print(f"  niveau {i} : {tuple(feat.shape)}")
    assert feat.shape[1] == 256, f"canaux != 256"

resolutions = [f.shape[-1] for f in out]
assert resolutions[0] > resolutions[1] > resolutions[2], \
    f"Résolutions non décroissantes : {resolutions}"
print(f"  Résolutions : {resolutions} ✅")

sum(f.sum() for f in out).backward()
for i, feat in enumerate(x):
    assert feat.grad is not None and feat.grad.abs().sum() > 0, \
        f"❌ Pas de gradient vers input {i}"
    print(f"  ✅ Gradient vers input {i} : |g|={feat.grad.abs().sum():.2f}")

print("\n✅ C.5 PASSÉ")

In [ ]:
# ═══ C.6 — Test wrapper complet ═══
import torch, sys, importlib
sys.path.insert(0, '/content')
for m in list(sys.modules.keys()):
    if m.startswith('mammo'): del sys.modules[m]
from mammo.backbone import DINOv2MultiScaleBackbone
from mammo.rfdetr_projector import MultiScaleProjector, BackboneProjectorWrapper

print("=" * 60); print("C.6 — Test BackboneProjectorWrapper"); print("=" * 60)

bb = DINOv2MultiScaleBackbone(checkpoint_path=DINOV2_CHECKPOINT,
                              freeze_blocks=2).cuda()
proj = MultiScaleProjector(
    in_channels=[384, 384, 384, 384],
    out_channels=256,
    scale_factors=(2.0, 1.0, 0.5, 0.25),
    num_blocks=3,
).cuda()
wrapper = BackboneProjectorWrapper(bb, proj).cuda()

print(f"  out_names   : {wrapper.out_names}")
print(f"  out_strides : {wrapper.out_strides}")
print(f"  output_shape: {wrapper.output_shape()}")
assert wrapper.out_names == ["p2", "p3", "p4", "p5"], \
    f"Attendu [p2,p3,p4,p5], obtenu {wrapper.out_names}"

wrapper.train()
x = torch.randn(1, 3, 812, 812, device='cuda')
out = wrapper(x)
for name, feat in out.items():
    print(f"  {name} : {tuple(feat.shape)}")
assert set(out.keys()) == {"p2", "p3", "p4", "p5"}

sum(f.sum() for f in out.values()).backward()
has_grad_3 = any(p.grad is not None and p.grad.abs().sum() > 0
                 for p in bb.vit.blocks[2].parameters())
assert has_grad_3, "❌ Pas de gradient au bloc 3"
print("  ✅ Gradient remonte jusqu'au bloc 3")

frozen_grad = any(p.grad is not None and p.grad.abs().sum() > 0
                  for p in bb.vit.blocks[0].parameters())
assert not frozen_grad
print("  ✅ Bloc 1 gelé")
print("\n✅ C.6 PASSÉ")

In [ ]:
# ═══ D.1 — Écriture mammo_pipeline.py (patché v2) ═══
%%writefile /content/mammo_pipeline.py
"""Pipeline mammo : mapper, register, optimizer + scheduler."""
import os, copy, re, json
from pathlib import Path
import cv2, numpy as np, torch
from detectron2.data import detection_utils as utils
import detectron2.data.transforms as T
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.data.datasets import load_coco_json
from detectron2.data.build import print_instances_class_histogram


def normalize_with_percentiles(img_16bit, low_pct=1.0, high_pct=99.0):
    if img_16bit is None or img_16bit.size == 0:
        return np.zeros_like(img_16bit, dtype=np.float32)
    h, w = img_16bit.shape[:2]
    step = max(1, min(h, w) // 300)
    sample = img_16bit[::step, ::step]
    fg = sample[sample > 0]
    target = fg if len(fg) >= 1000 else img_16bit[img_16bit > 0]
    if len(target) == 0:
        return np.zeros_like(img_16bit, dtype=np.float32)
    p_low, p_high = np.percentile(target, (low_pct, high_pct))
    if p_high <= p_low:
        return np.zeros_like(img_16bit, dtype=np.float32)
    return np.clip((img_16bit - p_low) / (p_high - p_low), 0.0, 1.0).astype(np.float32)


class Mammo16BitMapper:
    def __init__(self, augmentation=None, is_train=True,
                 img_format='RGB', keep_annotations=False,
                 images_fallback='/content/mammo_data/images',
                 use_percentile_norm=False):                 # ← FIX : flag explicite
        self.augmentation = augmentation or []
        self.is_train = is_train
        self.img_format = img_format
        self.keep_annotations = keep_annotations
        self.images_fallback = images_fallback
        self.use_percentile_norm = use_percentile_norm

    def _load_image_01(self, file_name):
        img = cv2.imread(file_name, cv2.IMREAD_UNCHANGED)
        if img is None and self.images_fallback:
            alt = os.path.join(self.images_fallback,
                               os.path.basename(file_name))
            if os.path.isfile(alt):
                img = cv2.imread(alt, cv2.IMREAD_UNCHANGED)
        if img is None:
            raise FileNotFoundError(f"❌ {file_name} (fallback : {self.images_fallback})")

        # ← FIX : normalisation par percentiles pour la mammo 16-bit
        if img.dtype == np.uint16:
            if self.use_percentile_norm:
                img = normalize_with_percentiles(img)          # → float32 [0,1]
            else:
                img = img.astype(np.float32) / 65535.0
        else:
            img = img.astype(np.float32) / 255.0

        if img.ndim == 2:
            img = np.stack([img] * 3, axis=-1)
        elif img.shape[2] == 1:
            img = np.repeat(img, 3, axis=2)
        return img

    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)
        image = self._load_image_01(dataset_dict["file_name"])
        utils.check_image_size(dataset_dict, image)
        image, transforms = T.apply_transform_gens(self.augmentation, image)
        image_shape = image.shape[:2]
        dataset_dict["image"] = torch.as_tensor(
            np.ascontiguousarray(image.transpose(2, 0, 1)))
        if not self.is_train and not self.keep_annotations:
            dataset_dict.pop("annotations", None)
            return dataset_dict
        if "annotations" in dataset_dict:
            annos = [
                utils.transform_instance_annotations(obj, transforms, image_shape)
                for obj in dataset_dict.pop("annotations")
                if obj.get("iscrowd", 0) == 0
            ]
            instances = utils.annotations_to_instances(annos, image_shape)
            dataset_dict["instances"] = utils.filter_empty_instances(instances)
        return dataset_dict


def _load_thing_classes(json_path):
    with open(json_path) as f:
        data = json.load(f)
    cats = data.get('categories')
    if not cats:
        raise ValueError(f"Aucune catégorie dans {json_path}")
    return [c['name'] for c in sorted(cats, key=lambda c: c['id'])]


def register_mammo_datasets(train_json, val_json, images_dir):
    for p, label in [(train_json, 'train_json'),
                     (val_json,   'val_json'),
                     (images_dir, 'images_dir')]:
        if not Path(p).exists():
            raise FileNotFoundError(f"{label} introuvable : {p}")

    thing_classes = _load_thing_classes(train_json)

    for name, json_file in [('mammo_train', train_json),
                            ('mammo_test',  val_json)]:
        if name in DatasetCatalog.list(): DatasetCatalog.remove(name)
        if name in MetadataCatalog.list(): MetadataCatalog.remove(name)
        DatasetCatalog.register(
            name,
            lambda j=json_file, d=images_dir, n=name: load_coco_json(j, d, n),
        )
        MetadataCatalog.get(name).set(
            thing_classes=thing_classes,
            json_file=json_file,
            image_root=images_dir,
        )
    print(f'✅ Datasets : {thing_classes}')


def print_dataset_summary(names=('mammo_train', 'mammo_test')):
    for name in names:
        dicts = DatasetCatalog.get(name)
        meta = MetadataCatalog.get(name)
        print(f"\n{'─'*60}\n📊 {name} : {len(dicts)} images\n{'─'*60}")
        print_instances_class_histogram(dicts, meta.thing_classes)


def make_dinov2_lr_groups(model, base_lr, backbone_lr, layer_decay,
                          weight_decay, num_layers=12,
                          backbone_prefix='backbone.backbone.vit.'):
    layer_groups = {}
    other_decay, other_no_decay = [], []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if name.startswith(backbone_prefix):
            m = re.search(r'blocks\.(\d+)\.', name)
            if m:
                depth = int(m.group(1)) + 1
                lr = backbone_lr * (layer_decay ** (num_layers - depth))
                layer_groups.setdefault(depth, {'params': [], 'lr': lr})
                layer_groups[depth]['params'].append(param)
            else:
                layer_groups.setdefault(0, {'params': [], 'lr': backbone_lr})
                layer_groups[0]['params'].append(param)
        else:
            if param.ndim == 1 or name.endswith('.bias'):
                other_no_decay.append(param)
            else:
                other_decay.append(param)

    groups = []
    for depth in sorted(layer_groups.keys()):
        g = layer_groups[depth]
        groups.append({
            'params': g['params'], 'lr': g['lr'],
            'weight_decay': 0.0, 'name': f'backbone_depth_{depth}',
        })
    if other_decay:
        groups.append({'params': other_decay, 'lr': base_lr,
                       'weight_decay': weight_decay, 'name': 'head_decay'})
    if other_no_decay:
        groups.append({'params': other_no_decay, 'lr': base_lr,
                       'weight_decay': 0.0, 'name': 'head_no_decay'})
    return groups


def build_mammo_optimizer(model, params_file='/content/train_params.json'):
    with open(params_file) as f:
        p = json.load(f)
    groups = make_dinov2_lr_groups(
        model,
        base_lr=p['LEARNING_RATE'],
        backbone_lr=p['BACKBONE_LR'],
        layer_decay=p['LAYER_DECAY'],
        weight_decay=p['WEIGHT_DECAY'],
    )
    print('📊 [Optimizer] Groupes :')
    for g in groups:
        n = sum(x.numel() for x in g['params'])
        print(f'   {g["name"]:>22s} | lr={g["lr"]:.2e} | '
              f'wd={g["weight_decay"]:.1e} | {n/1e6:.2f}M')
    return torch.optim.AdamW(
        groups, lr=p['LEARNING_RATE'], betas=(0.9, 0.999),
        weight_decay=p['WEIGHT_DECAY'],
    )


class WarmupMultiStepLR(torch.optim.lr_scheduler.LRScheduler):
    def __init__(self, optimizer, warmup_iters=2000, warmup_factor=1e-6,
                 warmup_method='linear', milestones=None, gamma=0.1,
                 last_epoch=-1):
        self.warmup_iters = warmup_iters
        self.warmup_factor = warmup_factor
        self.warmup_method = warmup_method
        self.milestones = sorted(milestones or [])
        self.gamma = gamma
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        if self.last_epoch < self.warmup_iters:
            alpha = self.last_epoch / max(1, self.warmup_iters)
            factor = (self.warmup_factor * (1 - alpha) + alpha
                      if self.warmup_method == 'linear'
                      else self.warmup_factor ** (1 - alpha))
        else:
            n_drops = sum(self.last_epoch >= m for m in self.milestones)
            factor = self.gamma ** n_drops
        return [base_lr * factor for base_lr in self.base_lrs]


def build_mammo_lr_scheduler(optimizer,
                              params_file='/content/train_params.json'):
    with open(params_file) as f:
        p = json.load(f)
    return WarmupMultiStepLR(
        optimizer,
        warmup_iters=p['WARMUP_ITERS'],
        warmup_factor=1e-6,
        warmup_method='linear',
        milestones=p['MILESTONES'],
        gamma=p['GAMMA'],
    )

In [ ]:
# ═══ D.2 — Test mammo_pipeline ═══
import sys, importlib, torch, numpy as np
sys.path.insert(0, '/content')
for m in list(sys.modules.keys()):
    if m.startswith('mammo') or m == 'mammo_pipeline': del sys.modules[m]
import mammo_pipeline; importlib.reload(mammo_pipeline)

print("=" * 60); print("D.2 — Test mammo_pipeline"); print("=" * 60)

from mammo_pipeline import normalize_with_percentiles
img = np.random.randint(0, 65535, (256, 256), dtype=np.uint16)
out = normalize_with_percentiles(img)
assert out.dtype == np.float32
assert 0.0 <= out.min() and out.max() <= 1.0
print(f"  normalize_with_percentiles : range=[{out.min():.3f}, {out.max():.3f}] ✅")

from mammo_pipeline import register_mammo_datasets, print_dataset_summary
register_mammo_datasets(TRAIN_JSON_DRIVE, TEST_JSON_DRIVE, IMAGES_DIR_LOCAL)
print_dataset_summary()

from detectron2.data import DatasetCatalog
assert 'mammo_train' in DatasetCatalog.list()
assert 'mammo_test' in DatasetCatalog.list()
print(f"  mammo_train : {len(DatasetCatalog.get('mammo_train'))} images")
print(f"  mammo_test  : {len(DatasetCatalog.get('mammo_test'))} images ✅")

import detectron2.data.transforms as T
from mammo_pipeline import Mammo16BitMapper
mapper = Mammo16BitMapper(
    augmentation=[T.ResizeShortestEdge(
        short_edge_length=(812,), max_size=1624, sample_style='choice')],
    is_train=False)
d = DatasetCatalog.get('mammo_test')[0]
out = mapper(d)
img = out['image']
assert img.shape[0] == 3, f"canaux : {img.shape[0]}"
assert img.dtype == torch.float32
assert 0.0 <= img.min() and img.max() <= 1.0
print(f"  Mapper : {tuple(img.shape)} range=[{img.min():.3f}, {img.max():.3f}] ✅")

from mammo_pipeline import WarmupMultiStepLR
opt = torch.optim.AdamW([torch.nn.Parameter(torch.zeros(1))], lr=1e-4)
sched = WarmupMultiStepLR(opt, warmup_iters=100, milestones=[500, 900], gamma=0.1)
lrs = []
for i in range(1100):
    sched.step()
    lrs.append(opt.param_groups[0]['lr'])

assert lrs[0] < lrs[99], "Warmup devrait augmenter le LR"
assert lrs[499] < lrs[498], "Milestone 500 devrait drop le LR"
assert lrs[899] < lrs[898], "Milestone 900 devrait drop le LR"
print(f"  Scheduler : lr[0]={lrs[0]:.2e}, lr[99]={lrs[99]:.2e}, "
      f"lr[499]={lrs[499]:.2e}, lr[899]={lrs[899]:.2e} ✅")

print("\n✅ D.2 PASSÉ")

In [ ]:
# ═══ E.1 — Écriture tensorboard_image_hook.py ═══
%%writefile /content/tensorboard_image_hook.py
"""Hook TensorBoard : log quelques prédictions pendant l'entraînement."""
import os
import torch
import numpy as np
from detectron2.engine import HookBase
from torch.utils.tensorboard import SummaryWriter


class TensorBoardImageHook(HookBase):
    def __init__(self, eval_loader, output_dir, period=500,
                 score_thresh=0.2, n_images=4, max_batches=8):
        self.loader = eval_loader
        self.output_dir = output_dir
        self.period = period
        self.score_thresh = score_thresh
        self.n_images = n_images
        self.max_batches = max_batches
        self.writer = None

    def before_train(self):
        tb_dir = os.path.join(self.output_dir, 'tensorboard_images')
        os.makedirs(tb_dir, exist_ok=True)
        self.writer = SummaryWriter(tb_dir)

    def after_step(self):
        if self.period <= 0 or self.writer is None:
            return
        if (self.trainer.iter + 1) % self.period != 0:
            return
        try:
            self._log()
        except Exception as e:
            import traceback
            print(f"⚠️ TensorBoardImageHook : {type(e).__name__}: {e}")
            traceback.print_exc()
            self.period = 0

    @torch.no_grad()
    def _log(self):
        import cv2
        model = self.trainer.model
        was_training = model.training
        model.eval()
        try:
            device = next(model.parameters()).device
            n = 0
            for bi, batch in enumerate(self.loader):
                if n >= self.n_images or bi >= self.max_batches:
                    break
                inputs = []
                for b in batch:
                    img = b["image"].to(device)
                    inputs.append({
                        "image": img,
                        "image_shape": tuple(img.shape[-2:]),
                        "height": int(img.shape[-2]),
                        "width": int(img.shape[-1]),
                    })
                outputs = model(inputs)
                if isinstance(outputs, dict):
                    outputs = [outputs]
                if not isinstance(outputs, (list, tuple)):
                    continue
                if len(outputs) != len(batch):
                    continue
                for b, o in zip(batch, outputs):
                    if n >= self.n_images:
                        break
                    if not isinstance(o, dict):
                        continue
                    img_t = b["image"].detach().cpu()
                    if img_t.ndim != 3:
                        continue
                    arr = img_t.numpy()
                    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)
                    vis = (arr.transpose(1, 2, 0) * 255).astype(np.uint8).copy()
                    inst = o.get("instances", None)
                    if inst is not None and inst.has("scores"):
                        scores = inst.scores
                        if scores.numel() > 0 and scores.ndim == 1:
                            keep = scores > self.score_thresh
                            if keep.any():
                                boxes = inst.pred_boxes[keep].tensor.cpu().numpy()
                                for (x1, y1, x2, y2) in boxes:
                                    cv2.rectangle(vis, (int(x1), int(y1)),
                                                  (int(x2), int(y2)), (255, 0, 0), 2)
                    self.writer.add_image(f'iter_{self.trainer.iter}/img_{n}',
                                          vis.transpose(2, 0, 1),
                                          self.trainer.iter)
                    n += 1
        finally:
            if was_training:
                model.train()

In [ ]:
# ═══ E.2 — Test import hook ═══
import sys, importlib
sys.path.insert(0, '/content')
for m in list(sys.modules.keys()):
    if m == 'tensorboard_image_hook': del sys.modules[m]
import tensorboard_image_hook; importlib.reload(tensorboard_image_hook)
from tensorboard_image_hook import TensorBoardImageHook
print("✅ E.2 — Import OK")

hook = TensorBoardImageHook(eval_loader=None, output_dir='/tmp/tb_test')
assert hook.period == 500
assert hook.score_thresh == 0.2
print("✅ E.2 PASSÉ")

In [ ]:
# ═══ F.1 — Écriture config (patché v2) ═══
%%writefile /content/mammo/configs/mammo_dinov2_dino.py
"""Config LazyConfig : DINOv2 ViT-S/14 + RF-DETR Projector + DINO."""
import os, sys, json, importlib.util

for _p in ('/content', '/content/detrex'):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import torch, torch.nn as nn
from detectron2.config import LazyCall as L
from detectron2.layers import ShapeSpec
from detrex.modeling.neck import ChannelMapper

from mammo.backbone import DINOv2MultiScaleBackbone
from mammo.rfdetr_projector import MultiScaleProjector, BackboneProjectorWrapper

_BASE = '/content/detrex/projects/dino/configs/models/dino_r50.py'
_spec = importlib.util.spec_from_file_location('_dino_r50_base', _BASE)
_base = importlib.util.module_from_spec(_spec)
sys.modules['_dino_r50_base'] = _base
_spec.loader.exec_module(_base)
model = _base.model

with open('/content/train_params.json') as f:
    _p = json.load(f)

model.backbone = L(BackboneProjectorWrapper)(
    backbone=L(DINOv2MultiScaleBackbone)(
        checkpoint_path=_p['DINOV2_CHECKPOINT'],
        model_name=_p['DINOV2_MODEL_NAME'],
        freeze_blocks=_p['FREEZE_BLOCKS'],
        out_features=("block3", "block6", "block9", "block12"),
    ),
    projector=L(MultiScaleProjector)(
        in_channels=[384, 384, 384, 384],
        out_channels=256,
        scale_factors=tuple(_p['RF_SCALE_FACTORS']),
        num_blocks=_p['RF_NUM_BLOCKS'],
        survival_prob=_p['RF_SURVIVAL_PROB'],
        force_drop_last_n_features=_p['RF_FORCE_DROP_N'],
    ),
)

model.neck = L(ChannelMapper)(
    input_shapes={
        'p2': ShapeSpec(channels=256, stride=7),
        'p3': ShapeSpec(channels=256, stride=14),
        'p4': ShapeSpec(channels=256, stride=28),
        'p5': ShapeSpec(channels=256, stride=56),
    },
    in_features=['p2', 'p3', 'p4', 'p5'],
    out_channels=256,
    num_outs=4,
    norm_layer=L(nn.GroupNorm)(num_groups=32, num_channels=256),
)

# --- Modèle ----------------------------------------------------------------
model.num_classes = 1

# ═══ FIX CRITIQUE : mutation par attribut (LazyCall = DictConfig OmegaConf) ═══
# model.kwargs[...] n'existe PAS → ConfigAttributeError.
# Les clés num_queries, dn_number, pixel_mean, pixel_std existent déjà
# dans dino_r50.py → la mutation par attribut fonctionne.
model.num_queries   = _p['NUM_QUERIES']
model.dn_number     = _p['NUM_DN_QUERIES']
model.pixel_mean    = [0.3192, 0.3192, 0.3192]
model.pixel_std     = [0.2603, 0.2603, 0.2603]

# select_box_nums_for_evaluation n'est PAS dans dino_r50.py → l'assignation
# crée la clé, qui sera passée à instantiate().
model.select_box_nums_for_evaluation = _p['NUM_QUERIES']

import detectron2.data as _d2data
import detectron2.data.transforms as T
from detectron2.data import build_detection_train_loader, build_detection_test_loader
from detectron2.evaluation import COCOEvaluator
from mammo_pipeline import Mammo16BitMapper

_train_mapper = Mammo16BitMapper(
    augmentation=[
        T.ResizeShortestEdge(
            short_edge_length=tuple(_p['IMAGE_SHORT_EDGES']),
            max_size=_p['IMAGE_MAX_SIZE'],
            sample_style='choice',
        ),
        T.RandomFlip(horizontal=True),
    ],
    is_train=True,
)
_test_mapper = Mammo16BitMapper(
    augmentation=[T.ResizeShortestEdge(
        short_edge_length=(int(_p['IMAGE_TEST_SIZE']),),
        max_size=_p['IMAGE_MAX_SIZE'],
        sample_style='choice',
    )],
    is_train=False,
)

dataloader = dict()
dataloader['train'] = L(build_detection_train_loader)(
    dataset=L(lambda: _d2data.DatasetCatalog.get('mammo_train'))(),
    mapper=_train_mapper,
    total_batch_size=_p['BATCH_SIZE'],
    num_workers=_p['NUM_WORKERS'],
)
dataloader['test'] = L(build_detection_test_loader)(
    dataset=L(lambda: _d2data.DatasetCatalog.get('mammo_test'))(),
    mapper=_test_mapper,
    num_workers=_p['NUM_WORKERS'],
)
dataloader['evaluator'] = L(COCOEvaluator)(
    dataset_name='mammo_test',
    output_dir=os.path.join(_p['OUTPUT_DIR'], 'eval'),
)

train = dict()
train['output_dir'] = _p['OUTPUT_DIR']
train['max_iter'] = _p['MAX_ITER']
train['log_period'] = _p['LOG_PERIOD']
train['eval_period'] = _p['ITERS_PER_EPOCH']
train['device'] = 'cuda'
train['amp'] = dict(enabled=_p['AMP_ENABLED'])
train['clip_grad'] = dict(
    enabled=True,
    params=dict(max_norm=_p['GRAD_CLIP_NORM'], norm_type=2.0),
)
train['checkpointer'] = dict(
    period=_p['CHECKPOINT_PERIOD'],
    max_to_keep=6,
)
seed = 42

In [ ]:
# ═══ F.2-bis — Validation empirique de la mutation LazyCall ═══
import sys, json
sys.path.insert(0, '/content'); sys.path.insert(0, '/content/detrex')
from detectron2.config import LazyConfig, instantiate
from mammo_pipeline import register_mammo_datasets

print("=" * 60); print("F.2-bis — Validation LazyCall"); print("=" * 60)

cfg = LazyConfig.load('/content/mammo/configs/mammo_dinov2_dino.py')

# 1. Valeurs côté LazyCall (avant instantiation)
print(f"  cfg.model.num_queries = {cfg.model.num_queries}")
print(f"  cfg.model.dn_number   = {cfg.model.dn_number}")
print(f"  type(cfg.model)       = {type(cfg.model).__name__}")

with open('/content/train_params.json') as f:
    _p = json.load(f)
assert cfg.model.num_queries == _p['NUM_QUERIES'], \
    f"❌ LazyCall non propagé : {cfg.model.num_queries}"
assert cfg.model.dn_number == _p['NUM_DN_QUERIES'], \
    f"❌ LazyCall non propagé : {cfg.model.dn_number}"
print("  ✅ LazyCall propagé")

# 2. Instantiation
register_mammo_datasets(_p['TRAIN_JSON_DRIVE'], _p['TEST_JSON_DRIVE'],
                        _p['LOCAL_IMAGES'])
model = instantiate(cfg.model)
assert model.num_queries == _p['NUM_QUERIES'], \
    f"❌ model.num_queries = {model.num_queries}"
assert model.dn_number == _p['NUM_DN_QUERIES'], \
    f"❌ model.dn_number = {model.dn_number}"
print(f"  ✅ model.num_queries={model.num_queries}, "
      f"model.dn_number={model.dn_number}")
print("\n✅ F.2-bis PASSÉ")

In [ ]:
# ═══ F.2 — Test config + instanciation ═══
import sys, importlib, json, torch
sys.path.insert(0, '/content'); sys.path.insert(0, '/content/detrex')
for m in list(sys.modules.keys()):
    if m.startswith('mammo') or m in ('mammo_pipeline', '_dino_r50_base'):
        del sys.modules[m]
import mammo, mammo.backbone, mammo.rfdetr_projector, mammo_pipeline
importlib.reload(mammo.backbone)
importlib.reload(mammo.rfdetr_projector)
importlib.reload(mammo_pipeline)

from detectron2.config import LazyConfig, instantiate
from mammo_pipeline import register_mammo_datasets

print("=" * 60)
print("F.2 — Test config + instanciation")
print("=" * 60)

# --- 1. Restaurer les constantes (idempotent après restart) ------------
with open('/content/train_params.json') as f:
    _p = json.load(f)
TRAIN_JSON_DRIVE = _p['TRAIN_JSON_DRIVE']
TEST_JSON_DRIVE  = _p['TEST_JSON_DRIVE']
IMAGES_DIR_LOCAL = _p['LOCAL_IMAGES']
DINOV2_CHECKPOINT = _p['DINOV2_CHECKPOINT']
os.makedirs(_p['OUTPUT_DIR'], exist_ok=True)

# --- 2. Enregistrer les datasets ---------------------------------------
register_mammo_datasets(TRAIN_JSON_DRIVE, TEST_JSON_DRIVE, IMAGES_DIR_LOCAL)

# --- 3. Charger le config ----------------------------------------------
cfg = LazyConfig.load('/content/mammo/configs/mammo_dinov2_dino.py')
print(f"  output_dir : {cfg.train.output_dir}")
print(f"  max_iter   : {cfg.train.max_iter}")
print(f"  num_queries: {cfg.model.num_queries}  (LazyCall)")
print(f"  dn_number  : {cfg.model.dn_number}  (LazyCall)")
print(f"  neck in_feat: {cfg.model.neck.in_features}")

# --- 4. Assertions sur le config (LazyCall) ----------------------------
assert list(cfg.model.neck.in_features) == ['p2', 'p3', 'p4', 'p5'], \
    f"❌ neck in_features = {cfg.model.neck.in_features}"
assert cfg.model.num_classes == 1, \
    f"❌ num_classes = {cfg.model.num_classes}"

# --- 5. Instanciation --------------------------------------------------
print("\n  Diagnostic LazyCall :")
print(f"    type(cfg.model) = {type(cfg.model).__name__}")
print(f"    cfg.model.num_queries = {getattr(cfg.model, 'num_queries', 'MISSING')}")
print(f"    cfg.model.dn_number   = {getattr(cfg.model, 'dn_number', 'MISSING')}")
if hasattr(cfg.model, '_kwargs'):
    print(f"    ._kwargs['num_queries'] = "
          f"{cfg.model._kwargs.get('num_queries', 'MISSING')}")
    print(f"    ._kwargs['dn_number']   = "
          f"{cfg.model._kwargs.get('dn_number', 'MISSING')}")

print("\n  Instanciation du modèle...")
model = instantiate(cfg.model)
print("  Modèle instancié ✅")

# --- 6. Assertions sur le MODÈLE INSTANCIÉ (source de vérité) ----------
assert model.num_queries == _p['NUM_QUERIES'], (
    f"❌ model.num_queries = {model.num_queries}, "
    f"attendu {_p['NUM_QUERIES']}. "
    f"Vérifier F.1 (mutation kwargs)."
)
assert model.dn_number == _p['NUM_DN_QUERIES'], (
    f"❌ model.dn_number = {model.dn_number}, "
    f"attendu {_p['NUM_DN_QUERIES']}. "
    f"Vérifier F.1 (utiliser ._kwargs si besoin)."
)
print(f"  ✅ num_queries={model.num_queries}, dn_number={model.dn_number}")

# --- 7. Statistiques de paramètres -------------------------------------
n_tot = sum(p.numel() for p in model.parameters())
n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Params totaux       : {n_tot/1e6:.2f}M")
print(f"  Params entraînables : {n_tr/1e6:.2f}M")

# --- 8. Forward eval ---------------------------------------------------
model = model.cuda().eval()
with torch.no_grad():
    outs = model([{
        "image": torch.randn(3, 812, 812, device='cuda'),
        "image_shape": (812, 812),
        "height": 812,
        "width": 812,
    }])
if isinstance(outs, dict):
    outs = [outs]
assert isinstance(outs, list) and len(outs) == 1, \
    f"❌ sortie : {type(outs)}"
assert outs[0].get("instances") is not None, \
    "❌ pas de 'instances' dans la sortie"
n_pred = outs[0]['instances'].pred_boxes.tensor.shape[0]
assert n_pred == _p['NUM_QUERIES'], (
    f"❌ {n_pred} prédictions, attendu {_p['NUM_QUERIES']}. "
    f"Vérifier select_box_nums_for_evaluation."
)
print(f"  Eval : {n_pred} prédictions ✅")

print("\n✅ F.2 PASSÉ")

In [ ]:
# ═══ G.1 — Écriture train_net.py (patché, complet) ═══
%%writefile /content/train_net.py
"""Entraînement DINO + DINOv2 SSL (RF-DETR style) avec grad accumulation + bf16."""
import os, sys, json, logging, warnings

# --- Warnings ciblés (ne masque que ce qui est nécessaire) -------------
warnings.filterwarnings('ignore', category=FutureWarning,
                        module=r'.*timm\.models\.layers.*')
warnings.filterwarnings('ignore', category=FutureWarning,
                        module=r'.*detectron2\.layers\.dcn_v3.*')
warnings.filterwarnings('ignore', category=FutureWarning,
                        message=r'.*torch\.cuda\.amp\.custom_.*')
warnings.filterwarnings('ignore', category=UserWarning,
                        message=r'.*pkg_resources is deprecated.*')

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
sys.path.insert(0, '/content')
sys.path.insert(0, '/content/detrex')

import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from detectron2.config import LazyConfig, instantiate
from detectron2.engine import (SimpleTrainer, default_argument_parser,
                                default_setup, hooks, HookBase)
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.utils.events import (CommonMetricPrinter, JSONWriter,
                                      TensorboardXWriter)
from detectron2.evaluation import inference_on_dataset

from tensorboard_image_hook import TensorBoardImageHook
from mammo_pipeline import (register_mammo_datasets,
                             build_mammo_optimizer,
                             build_mammo_lr_scheduler)


class BestModelHook(HookBase):
    """Sauvegarde le meilleur modèle selon une métrique."""

    def __init__(self, metric='bbox/AP50', min_max='max'):
        self.metric = metric
        self.min_max = min_max
        self.best_value = float('-inf') if min_max == 'max' else float('inf')
        self.best_iter = -1
        self._logger = logging.getLogger('detectron2.best_model')
        self._logger.setLevel(logging.INFO)
        if not self._logger.handlers:
            h = logging.StreamHandler()
            h.setFormatter(logging.Formatter('[%(name)s] %(message)s'))
            self._logger.addHandler(h)
        self._logger.propagate = False

    def before_train(self):
        out = getattr(self.trainer, 'checkpointer', None)
        if out:
            meta = os.path.join(out.save_dir, 'best_metric.json')
            if os.path.isfile(meta):
                try:
                    with open(meta) as f:
                        s = json.load(f)
                    self.best_value = float(s['value'])
                    self.best_iter = int(s.get('best_iter', -1))
                    self._logger.info(f'🔄 Reprise : {self.best_value:.4f}')
                except Exception as e:
                    self._logger.warning(f'⚠️ {e}')

    def after_step(self):
        latest = self.trainer.storage.latest()
        val, it, key = None, None, None
        for k in [self.metric, 'bbox/AP50', 'bbox/AP', 'AP50', 'AP']:
            if k in latest:
                val, it = latest[k]
                key = k
                break
        if val is None or it <= self.best_iter:
            return
        better = ((self.min_max == 'max' and val > self.best_value) or
                  (self.min_max == 'min' and val < self.best_value))
        if better:
            self._logger.info(f'🏆 iter {it} : {key} = {val:.4f}')
            self.best_value, self.best_iter = val, it
            if hasattr(self.trainer, 'checkpointer'):
                # 'best' ne collisionne PAS avec 'ckpt_XXXX' de PeriodicCheckpointer
                self.trainer.checkpointer.save('best')
            try:
                meta = os.path.join(self.trainer.checkpointer.save_dir,
                                    'best_metric.json')
                with open(meta + '.tmp', 'w') as f:
                    json.dump({'best_iter': self.best_iter, 'metric': key,
                               'value': float(val)}, f, indent=2)
                os.replace(meta + '.tmp', meta)
            except Exception as e:
                self._logger.warning(f'⚠️ {e}')


class DetrexTrainer(SimpleTrainer):
    """Trainer avec grad accumulation + support bf16/fp16."""

    def __init__(self, model, dataloader, optimizer, amp=False,
                 clip_grad_params=None, grad_scaler=None,
                 grad_accum_steps=1, amp_dtype=None):
        super().__init__(model=model, data_loader=dataloader,
                         optimizer=optimizer)
        self.amp = amp
        self.clip_grad_params = clip_grad_params
        self.grad_accum_steps = max(1, int(grad_accum_steps))

        self.amp_dtype = torch.bfloat16 if amp_dtype is None else amp_dtype

        # GradScaler uniquement utile en fp16 (bf16 n'en a pas besoin)
        self.grad_scaler = None
        if amp and self.amp_dtype == torch.float16:
            if grad_scaler is None:
                from torch.cuda.amp import GradScaler
                grad_scaler = GradScaler()
            self.grad_scaler = grad_scaler

    def _should_step(self):
        it = self.iter + 1
        max_iter = getattr(self, 'max_iter', None)
        if max_iter is not None and it >= max_iter:
            return True
        return it % self.grad_accum_steps == 0

    def run_step(self):
        import time
        from torch.amp import autocast
        self.model.train()
        start = time.perf_counter()
        data = next(self._data_loader_iter)
        data_time = time.perf_counter() - start

        accum = self.grad_accum_steps
        do_step = self._should_step()

        if self.amp:
            with autocast('cuda', dtype=self.amp_dtype):
                loss_dict = self.model(data)
                if isinstance(loss_dict, torch.Tensor):
                    loss_dict = {"total_loss": loss_dict}
                elif not isinstance(loss_dict, dict):
                    raise RuntimeError(f"Model returned {type(loss_dict)}")
                losses = sum(loss_dict.values()) / accum

            if self.grad_scaler is not None:
                # fp16 : avec scaler
                self.grad_scaler.scale(losses).backward()
                if do_step:
                    if self.clip_grad_params is not None:
                        self.grad_scaler.unscale_(self.optimizer)
                        self.clip_grads(self.model.parameters())
                    self.grad_scaler.step(self.optimizer)
                    self.grad_scaler.update()
                    self.optimizer.zero_grad()
            else:
                # bf16 : sans scaler
                losses.backward()
                if do_step:
                    if self.clip_grad_params is not None:
                        self.clip_grads(self.model.parameters())
                    self.optimizer.step()
                    self.optimizer.zero_grad()
        else:
            loss_dict = self.model(data)
            if isinstance(loss_dict, torch.Tensor):
                loss_dict = {"total_loss": loss_dict}
            elif not isinstance(loss_dict, dict):
                raise RuntimeError(f"Model returned {type(loss_dict)}")
            losses = sum(loss_dict.values()) / accum
            losses.backward()
            if do_step:
                if self.clip_grad_params is not None:
                    self.clip_grads(self.model.parameters())
                self.optimizer.step()
                self.optimizer.zero_grad()

        # Re-scale pour le logging (report la vraie loss du microbatch)
        loss_log = {k: (v * accum if isinstance(v, torch.Tensor) else v)
                    for k, v in loss_dict.items()}
        self._write_metrics(loss_log, data_time)

    def clip_grads(self, params):
        params = [p for p in params
                  if p.requires_grad and p.grad is not None]
        if params:
            tn = torch.nn.utils.clip_grad_norm_(params, **self.clip_grad_params)
            self.storage.put_scalar("grad_norm", tn.item())

    def state_dict(self):
        ret = super().state_dict()
        if self.grad_scaler is not None:
            ret["grad_scaler"] = self.grad_scaler.state_dict()
        return ret

    def load_state_dict(self, sd):
        super().load_state_dict(sd)
        if self.grad_scaler is not None and "grad_scaler" in sd:
            self.grad_scaler.load_state_dict(sd["grad_scaler"])


def do_train(args, cfg):
    p = {}
    if os.path.exists('/content/train_params.json'):
        with open('/content/train_params.json') as f:
            p = json.load(f)

    register_mammo_datasets(p.get('TRAIN_JSON_DRIVE'),
                            p.get('TEST_JSON_DRIVE'),
                            p.get('LOCAL_IMAGES'))

    # --- Modèle --------------------------------------------------------
    model = instantiate(cfg.model)
    device = torch.device(cfg.train.device
                          if hasattr(cfg.train, 'device') else 'cuda')
    model.to(device)

    # --- Sanity check (le modèle DOIT avoir les bonnes valeurs) --------
    assert model.dn_number == p['NUM_DN_QUERIES'], \
        f"❌ model.dn_number = {model.dn_number}, " \
        f"attendu {p['NUM_DN_QUERIES']}"
    assert model.num_queries == p['NUM_QUERIES'], \
        f"❌ model.num_queries = {model.num_queries}, " \
        f"attendu {p['NUM_QUERIES']}"

    n_bb = sum(p_.numel() for n, p_ in model.named_parameters()
               if p_.requires_grad and n.startswith('backbone'))
    n_tot = sum(p_.numel() for p_ in model.parameters() if p_.requires_grad)
    print(f'📊 Backbone : {n_bb/1e6:.2f} M | Total : {n_tot/1e6:.2f} M')
    print(f'📊 num_queries={model.num_queries}, dn_number={model.dn_number}')

    # --- Optimizer (layer-wise LR, sans cfg) ---------------------------
    optim = build_mammo_optimizer(model)
    train_loader = instantiate(cfg.dataloader.train)

    # --- Clip grad -----------------------------------------------------
    clip_grad_params = None
    if getattr(getattr(cfg.train, 'clip_grad', None), 'enabled', False):
        clip_grad_params = cfg.train.clip_grad.params

    # --- AMP -----------------------------------------------------------
    grad_accum = int(p.get('GRAD_ACCUM_STEPS', 1))
    amp_enabled = getattr(getattr(cfg.train, 'amp', None), 'enabled', False)
    amp_dtype_str = p.get('AMP_DTYPE', 'bfloat16')
    amp_dtype = {'bfloat16': torch.bfloat16,
                 'float16': torch.float16}.get(amp_dtype_str, torch.bfloat16)

    print(f'🔁 Grad accumulation : {grad_accum} '
          f'(batch effectif = {p.get("BATCH_SIZE", 1) * grad_accum})')
    print(f'⚡ AMP : {amp_enabled} ({amp_dtype_str})')

    # --- Trainer -------------------------------------------------------
    trainer = DetrexTrainer(
        model=model, dataloader=train_loader, optimizer=optim,
        amp=amp_enabled,
        clip_grad_params=clip_grad_params,
        grad_accum_steps=grad_accum,
        amp_dtype=amp_dtype,
    )
    trainer.max_iter = cfg.train.max_iter

    lr_sched = build_mammo_lr_scheduler(optim)

    # --- Checkpointer --------------------------------------------------
    checkpointer = DetectionCheckpointer(
        model, cfg.train.output_dir,
        optimizer=optim,
        scheduler=lr_sched,
        trainer=trainer,
    )
    trainer.checkpointer = checkpointer

    # --- Hooks ---------------------------------------------------------
    extra_hooks = [
        hooks.IterationTimer(),
        hooks.LRScheduler(optimizer=optim, scheduler=lr_sched),
        hooks.PeriodicCheckpointer(
            checkpointer,
            period=cfg.train.checkpointer.period,
            max_to_keep=cfg.train.checkpointer.max_to_keep,
            file_prefix='ckpt',  # ← FIX : 'ckpt' évite la collision avec 'best'
        ),
    ]

    if 'evaluator' in cfg.dataloader:
        eval_loader = instantiate(cfg.dataloader.test)
        evaluator = instantiate(cfg.dataloader.evaluator)
        extra_hooks.append(hooks.EvalHook(
            cfg.train.eval_period,
            lambda: inference_on_dataset(model, eval_loader, evaluator),
        ))
        extra_hooks.append(BestModelHook(metric='bbox/AP50', min_max='max'))
        extra_hooks.append(TensorBoardImageHook(
            eval_loader, output_dir=cfg.train.output_dir,
            period=p.get('TB_IMAGE_PERIOD', 500),
            score_thresh=p.get('SCORE_THRESH', 0.20),
        ))
    else:
        extra_hooks.append(BestModelHook())

    # --- Writers -------------------------------------------------------
    tb_dir = os.path.join(cfg.train.output_dir, 'tensorboard')
    os.makedirs(tb_dir, exist_ok=True)
    writers = [
        CommonMetricPrinter(cfg.train.max_iter),
        JSONWriter(os.path.join(cfg.train.output_dir, 'metrics.json')),
        TensorboardXWriter(tb_dir),
    ]
    extra_hooks.append(hooks.PeriodicWriter(writers,
                                             period=cfg.train.log_period))
    trainer.register_hooks(extra_hooks)

    # --- Resume --------------------------------------------------------
    start_iter = checkpointer.resume_or_load(
        args.resume_from or '', resume=args.resume
    ).get('iteration', -1) + 1

    # --- FIX scheduler : resync si l'état n'a pas été restauré ---------
    if lr_sched.last_epoch < start_iter - 1:
        print(f"⚠️  Scheduler non restauré : "
              f"last_epoch={lr_sched.last_epoch} < "
              f"start_iter-1={start_iter - 1}")
        print(f"    Resynchronisation à last_epoch={start_iter - 1}...")
        lr_sched.last_epoch = start_iter - 1
        if hasattr(lr_sched, '_step_count'):
            lr_sched._step_count = start_iter
        new_lrs = lr_sched.get_lr()
        for pg, lr in zip(optim.param_groups, new_lrs):
            pg['lr'] = lr
        print(f"    ✅ LR après resync : "
              f"head={optim.param_groups[0]['lr']:.2e} | "
              f"backbone_top={optim.param_groups[-2]['lr']:.2e}")

    # --- Go ------------------------------------------------------------
    print(f'🚀 Entraînement iter {start_iter} → {cfg.train.max_iter}')
    trainer.train(start_iter, cfg.train.max_iter)
    print('✅ Terminé')


def main():
    parser = default_argument_parser()
    parser.set_defaults(
        config_file='/content/mammo/configs/mammo_dinov2_dino.py')
    parser.add_argument('--resume-from', default='')
    args = parser.parse_args()
    cfg = LazyConfig.load(args.config_file)
    if args.opts:
        cfg = LazyConfig.apply_overrides(cfg, args.opts)
    os.makedirs(cfg.train.output_dir, exist_ok=True)
    default_setup(cfg, args)
    do_train(args, cfg)


if __name__ == '__main__':
    main()

In [ ]:
# ═══ H.1 — Smoke test complet ═══
import sys, importlib, torch, json
sys.path.insert(0, '/content'); sys.path.insert(0, '/content/detrex')

# --- Recharger proprement ----------------------------------------------
for m in list(sys.modules.keys()):
    if m.startswith('mammo') or m in ('mammo_pipeline', 'train_net',
                                       '_dino_r50_base',
                                       'tensorboard_image_hook'):
        del sys.modules[m]
import mammo.backbone, mammo.rfdetr_projector, mammo_pipeline, train_net
importlib.reload(mammo.backbone)
importlib.reload(mammo.rfdetr_projector)
importlib.reload(mammo_pipeline)
importlib.reload(train_net)

from detectron2.config import LazyConfig, instantiate
from mammo_pipeline import register_mammo_datasets, Mammo16BitMapper
import detectron2.data.transforms as T
from detectron2.data import DatasetCatalog

print("=" * 60); print("H.1 — Smoke test complet"); print("=" * 60)

# --- 1. Constantes idempotentes ----------------------------------------
with open('/content/train_params.json') as f:
    _p = json.load(f)
TRAIN_JSON_DRIVE  = _p['TRAIN_JSON_DRIVE']
TEST_JSON_DRIVE   = _p['TEST_JSON_DRIVE']
IMAGES_DIR_LOCAL  = _p['LOCAL_IMAGES']
DINOV2_CHECKPOINT = _p['DINOV2_CHECKPOINT']

# --- 2. Enregistrer + charger config -----------------------------------
register_mammo_datasets(TRAIN_JSON_DRIVE, TEST_JSON_DRIVE, IMAGES_DIR_LOCAL)
cfg = LazyConfig.load('/content/mammo/configs/mammo_dinov2_dino.py')
model = instantiate(cfg.model).cuda()

# --- 3. Récupérer un échantillon (avec garde) --------------------------
mapper_train = Mammo16BitMapper(
    augmentation=[T.ResizeShortestEdge(
        short_edge_length=(700,), max_size=1624, sample_style='choice')],
    is_train=True,
    images_fallback=IMAGES_DIR_LOCAL,
)

train_dicts = DatasetCatalog.get('mammo_train')
assert len(train_dicts) > 0, "❌ Dataset mammo_train vide"
d = train_dicts[0]

sample = mapper_train(d)
assert sample['image'].shape[0] == 3, \
    f"❌ canaux = {sample['image'].shape[0]}"
assert 'instances' in sample, "❌ pas d'instances dans l'échantillon"
assert len(sample['instances']) > 0, "❌ 0 instances dans le premier sample"

batch = [sample]
print(f"  Batch : image {tuple(sample['image'].shape)}, "
      f"{len(sample['instances'])} instances")

# --- 4. Forward train --------------------------------------------------
model.train()
loss_dict = model(batch)
assert isinstance(loss_dict, dict), \
    f"❌ attendu dict, obtenu {type(loss_dict)}"

print(f"  Losses ({len(loss_dict)} clés) :")
for k, v in loss_dict.items():
    assert torch.isfinite(v), f"❌ {k} = {v} (non fini)"
    print(f"    {k} : {v.item():.4f}")

# --- 5. Vérifier présence de DN losses (validation implicite dn_number) -
dn_keys = [k for k in loss_dict.keys() if 'dn' in k]
assert len(dn_keys) > 0, \
    "❌ Aucune loss DN — dn_number=0 ou DN désactivé"
print(f"  ✅ DN losses présentes ({len(dn_keys)} clés)")

# --- 6. Backward -------------------------------------------------------
total_loss = sum(loss_dict.values())
assert torch.isfinite(total_loss), f"❌ total_loss = {total_loss}"
total_loss.backward()
print(f"  ✅ Backward OK, total={total_loss.item():.4f}")

# --- 7. Gradient sur le projector --------------------------------------
proj_params = [(n, p) for n, p in model.named_parameters()
               if 'projector' in n and p.requires_grad and p.grad is not None]
assert len(proj_params) > 0, "❌ Aucun gradient sur le projector"
print(f"  ✅ {len(proj_params)} params projector ont un gradient")

# --- 8. Gradient sur le neck -------------------------------------------
neck_params = [(n, p) for n, p in model.named_parameters()
               if 'neck' in n and p.requires_grad and p.grad is not None]
assert len(neck_params) > 0, "❌ Aucun gradient sur le neck"
print(f"  ✅ {len(neck_params)} params neck ont un gradient")

# --- 9. Gradient sur le décodeur ---------------------------------------
dec_keys = ('decoder', 'transformer', 'encoder')
dec_params = [(n, p) for n, p in model.named_parameters()
              if any(k in n for k in dec_keys)
              and p.requires_grad and p.grad is not None]
print(f"  ✅ {len(dec_params)} params encoder/decoder ont un gradient")

# --- 10. Optimizer step ------------------------------------------------
optim = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=1e-5)
optim.step()
print(f"  ✅ Optimizer step OK")

print("\n✅ H.1 PASSÉ")

In [ ]:
# ═══ H.2 — Mini-run (8 iters, accum=4, AMP off) — patché ═══
import sys, importlib, torch, json, os, time
sys.path.insert(0, '/content'); sys.path.insert(0, '/content/detrex')

# --- Recharger proprement ----------------------------------------------
for m in list(sys.modules.keys()):
    if m.startswith('mammo') or m in ('mammo_pipeline', 'train_net',
                                       '_dino_r50_base',
                                       'tensorboard_image_hook'):
        del sys.modules[m]
import mammo.backbone, mammo.rfdetr_projector, mammo_pipeline, train_net
importlib.reload(mammo.backbone)
importlib.reload(mammo.rfdetr_projector)
importlib.reload(mammo_pipeline)
importlib.reload(train_net)

from detectron2.config import LazyConfig, instantiate
from mammo_pipeline import register_mammo_datasets, build_mammo_optimizer

print("=" * 60); print("H.2 — Mini-run (8 iters, accum=4)"); print("=" * 60)

PARAMS = '/content/train_params.json'

# --- 1. Backup AVANT toute mutation ------------------------------------
with open(PARAMS) as f:
    p_orig = json.load(f)
print(f"  💾 Backup train_params.json : "
      f"accum={p_orig['GRAD_ACCUM_STEPS']}, "
      f"AMP={p_orig['AMP_ENABLED']}")

try:
    # --- 2. Patch temporaire -------------------------------------------
    p = dict(p_orig)
    p['AMP_ENABLED'] = False
    p['GRAD_ACCUM_STEPS'] = 4
    with open(PARAMS, 'w') as f:
        json.dump(p, f, indent=2)
    print(f"  ⚙️  Patch : AMP={p['AMP_ENABLED']} | "
          f"accum={p['GRAD_ACCUM_STEPS']}")

    # --- 3. Setup ------------------------------------------------------
    _p = p  # alias
    TRAIN_JSON_DRIVE = _p['TRAIN_JSON_DRIVE']
    TEST_JSON_DRIVE  = _p['TEST_JSON_DRIVE']
    IMAGES_DIR_LOCAL = _p['LOCAL_IMAGES']

    register_mammo_datasets(TRAIN_JSON_DRIVE, TEST_JSON_DRIVE,
                            IMAGES_DIR_LOCAL)
    cfg = LazyConfig.load('/content/mammo/configs/mammo_dinov2_dino.py')
    model = instantiate(cfg.model).cuda()

    # Sanity check : dn_number/num_queries bien propagés
    assert model.dn_number == _p['NUM_DN_QUERIES'], \
        f"❌ model.dn_number = {model.dn_number}"
    assert model.num_queries == _p['NUM_QUERIES'], \
        f"❌ model.num_queries = {model.num_queries}"
    print(f"  ✅ model.num_queries={model.num_queries}, "
          f"dn_number={model.dn_number}")

    # --- 4. Optimizer + loader -----------------------------------------
    optim = build_mammo_optimizer(model)
    loader = instantiate(cfg.dataloader.train)

    # --- 5. Trainer ----------------------------------------------------
    trainer = train_net.DetrexTrainer(
        model=model, dataloader=loader, optimizer=optim,
        amp=_p['AMP_ENABLED'],
        clip_grad_params=dict(max_norm=_p['GRAD_CLIP_NORM'], norm_type=2.0),
        grad_accum_steps=_p['GRAD_ACCUM_STEPS'],
        amp_dtype=torch.bfloat16,
    )

    # --- 6. Boucle de test ---------------------------------------------
    from detectron2.utils.events import EventStorage
    with EventStorage() as storage:
        trainer.storage = storage
        trainer.register_hooks([train_net.hooks.IterationTimer()])
        trainer.before_train()
        t0 = time.time()
        for i in range(8):
            trainer.iter = i
            trainer.run_step()
            latest = storage.latest()
            if 'grad_norm' in latest:
                gn = latest['grad_norm'][0]
                print(f"  iter {i+1}/8 — grad_norm={gn:.4f}")
            else:
                print(f"  iter {i+1}/8 — grad_norm=(accum)")
        trainer.after_train()
        elapsed = time.time() - t0

    print(f"  ✅ 8 iters en {elapsed:.1f}s ({elapsed/8:.2f}s/iter)")

finally:
    # --- 7. Restauration INCONDITIONNELLE ------------------------------
    with open(PARAMS, 'w') as f:
        json.dump(p_orig, f, indent=2)
    print(f"  ♻️  train_params.json restauré : "
          f"accum={p_orig['GRAD_ACCUM_STEPS']}, "
          f"AMP={p_orig['AMP_ENABLED']}")

    # --- 8. Vérification post-restauration -----------------------------
    with open(PARAMS) as f:
        p_check = json.load(f)
    assert p_check['GRAD_ACCUM_STEPS'] == p_orig['GRAD_ACCUM_STEPS'], \
        f"❌ Restauration échouée : accum={p_check['GRAD_ACCUM_STEPS']}"
    assert p_check['AMP_ENABLED'] == p_orig['AMP_ENABLED'], \
        f"❌ Restauration échouée : AMP={p_check['AMP_ENABLED']}"
    print(f"  ✅ Restauration vérifiée")

print("\n✅ H.2 PASSÉ — Prêt pour le run complet")

In [ ]:
# ═══ I.1 — Lancement du run complet ═══
import sys
sys.path.insert(0, '/content'); sys.path.insert(0, '/content/detrex')

# Recharger proprement train_net et dépendances
import importlib
for m in list(sys.modules.keys()):
    if m.startswith('mammo') or m in ('mammo_pipeline', 'train_net',
                                       '_dino_r50_base', 'tensorboard_image_hook'):
        del sys.modules[m]
import mammo_pipeline, train_net
importlib.reload(mammo_pipeline)
importlib.reload(train_net)

# Vérifier la config finale
import json
with open('/content/train_params.json') as f:
    p = json.load(f)
print("=" * 60)
print("🚀 LANCEMENT DU RUN COMPLET")
print("=" * 60)
print(f"  MAX_ITER       = {p['MAX_ITER']:,}")
print(f"  EPOCHS         = {p['NUM_EPOCHS']}")
print(f"  BATCH × ACCUM  = {p['BATCH_SIZE']} × {p['GRAD_ACCUM_STEPS']} = "
      f"{p['BATCH_SIZE']*p['GRAD_ACCUM_STEPS']}")
print(f"  AMP            = {p['AMP_ENABLED']} ({p['AMP_DTYPE']})")
print(f"  MILESTONES     = {p['MILESTONES']}")
print(f"  OUTPUT_DIR     = {p['OUTPUT_DIR']}")
print("=" * 60)

sys.argv = [
    'train_net.py',
    '--config-file', '/content/mammo/configs/mammo_dinov2_dino.py',
    '--num-gpus', '1',
]
train_net.main()

In [ ]:
# ═══ I.2 — Reprise depuis le dernier checkpoint ═══
import sys
sys.argv = [
    'train_net.py',
    '--config-file', '/content/mammo/configs/mammo_dinov2_dino.py',
    '--num-gpus', '1',
    '--resume',   # ← reprend automatiquement depuis OUTPUT_DIR
]
import train_net
train_net.main()